# Thesis workflow
Select a GPU runtime for full evaluation or training. The original training used an NVIDIA L4.


In [ ]:
from pathlib import Path
import subprocess, sys
SOURCE = 'github'  # 'github' or 'upload'
if SOURCE == 'github':
    checkout = Path('/content/thesis-volatility-neural-lsv')
    if not checkout.exists():
        subprocess.run(['git','clone','https://github.com/denistar490-png/thesis-volatility-neural-lsv.git',str(checkout)],check=True)
    project = checkout/'thesis_simple'
else:
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    archive = next(name for name in uploaded if name.endswith('.zip'))
    destination = Path('/content/simple_submission').resolve()
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            if not (destination/member.filename).resolve().is_relative_to(destination):
                raise ValueError('Invalid ZIP member path')
        z.extractall(destination)
    project = next(destination.rglob('run.py')).parent
assert (project/'inputs/trained_model.pt').is_file(), 'Push thesis_simple to GitHub first, or upload the new ZIP.'
subprocess.run([sys.executable,'-m','pip','install','-r',str(project/'requirements.txt')],check=True)


## Short execution check
Small samples check that the code executes. They do not replace the full thesis estimates. The result directory must be new for each run.


In [ ]:
subprocess.run([sys.executable,str(project/'run.py'),'reproduce','--profile','smoke','--output','/content/thesis_simple_smoke'],cwd=project,check=True)


## Full reproduction or new training
Choose `reproduce` to use the supplied network or `train` to train from the beginning. Completed training stages are saved to Drive. To resume, keep the run name and set `RESUME=True`, an interrupted maturity restarts.


In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime before a full run.'
COMMAND = 'reproduce'  # or 'train'
RUN_NAME = 'simple_reproduction_01'
RESUME = False
output = Path('/content/drive/MyDrive/thesis_simple_runs')/RUN_NAME
args = [sys.executable,str(project/'run.py'),COMMAND,'--profile','thesis','--device','cuda','--output',str(output)]
if RESUME:
    assert COMMAND == 'train'
    args.append('--resume')
subprocess.run(args,cwd=project,check=True)


In [ ]:
import pandas as pd
display(pd.read_csv(output/'comparison_overall.csv'))
print('Figures:',output/'figures')
